# Autonomous Drone Navigation using Deep Reinforcement Learning

This notebook documents an AirSim-based reinforcement learning project for autonomous drone navigation. The task is to train a drone agent to use visual observations and motion state to avoid obstacles and reach a target position efficiently.

## 1. Introduction, Motivation, and Problem Statement

Autonomous drone navigation requires real-time decision-making under uncertain visual conditions. A drone must interpret its surroundings, avoid obstacles, maintain a safe flight altitude, and move toward a designated target. Classical planning methods can work well in known maps, but they often require explicit obstacle models and may struggle when the environment changes.

The purpose of this project is to develop and evaluate deep reinforcement learning systems for drone navigation in Microsoft AirSim. The agent learns a navigation policy from interaction with the simulator. The project compares discrete-action DQN and PPO baselines that combine depth-camera input with low-dimensional kinematic features.

## 2. Data Source and Reinforcement Learning Task

This project does not use a fixed supervised learning dataset. Instead, data is generated online by the AirSim simulator. At each time step, the drone receives an observation, chooses an action, and receives a reward from the environment.

| Component | Definition |
|---|---|
| Simulator | Microsoft AirSim Blocks environment |
| Observation | Depth image from the front camera plus relative target and velocity features |
| Action space | 6 discrete actions: forward, left, right, up, down, hover |
| Reward | Positive reward for progress toward the target, large reward for reaching the goal, penalties for collisions, unsafe altitude, and time steps |
| Episode termination | Goal reached, collision, unsafe altitude, or maximum step limit |
| Evaluation metrics | success rate, collision rate, average reward, final distance, average episode length |

In [ ]:
from pathlib import Path
import csv
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT

## 3. Exploratory Analysis of the RL Task

The RL task is challenging because observations are high-dimensional, rewards can be sparse, and unsafe actions can quickly lead to collisions. Before training, it is important to inspect the observation shapes, action definitions, target location, and termination criteria.

In [ ]:
from airsim_drone_env import DroneEnvConfig

config = DroneEnvConfig()
task_summary = {
    "image_shape": (1, config.image_height, config.image_width),
    "state_features": ["relative_x", "relative_y", "relative_z", "velocity_x", "velocity_y", "velocity_z"],
    "num_actions": 6,
    "target_position": config.target_position,
    "goal_radius_m": config.goal_radius_m,
    "max_steps": config.max_steps,
}
task_summary

### AirSim Observation Inspection

The following cell requires the Blocks simulator to be running. It resets the drone environment and displays the observation dimensions. If AirSim is not running, skip this cell and return to it after starting `Blocks.exe`.

In [ ]:
# Optional: run only after starting D:\\AirSim\\Blocks\\WindowsNoEditor\\Blocks.exe
# from airsim_drone_env import AirSimDroneEnv
# env = AirSimDroneEnv(config)
# obs, info = env.reset()
# print("Image shape:", obs["image"].shape)
# print("State shape:", obs["state"].shape)
# print("Initial info:", info)
# env.close()

### Challenging Aspects

- The visual input is high-dimensional compared with the small number of actions.
- The drone must balance exploration with safety because random actions can cause collisions.
- Rewards are partly sparse: the largest positive reward only occurs when the drone reaches the target.
- AirSim uses the NED coordinate system, where negative `z` means higher altitude.
- Policies trained in one environment may not generalize to different obstacle layouts without further randomization or training.

## 4. Models and Methods

The project uses two deep reinforcement learning baselines: Deep Q-Network (DQN) and Proximal Policy Optimization (PPO). DQN is appropriate because the action space is discrete and it is a standard value-based baseline for visual control tasks. PPO is included as a policy-gradient comparison method with a clipped objective that often provides more stable updates than vanilla policy gradients.

The model consists of:

- a convolutional encoder for the depth image;
- a small vector of kinematic features containing relative target position and velocity;
- a DQN Q-value head or PPO actor-critic heads for discrete flight actions.

The DQN training procedure uses experience replay, epsilon-greedy exploration, a target network, and Huber loss. These design choices follow the core ideas of Mnih et al. (2015), adapted to the AirSim drone navigation environment. The PPO training procedure uses sampled rollouts, generalized advantage estimation, clipped policy updates, entropy regularisation, and a value-function loss, following Schulman et al. (2017).

In [ ]:
from dqn_agent import DQNConfig, DQNAgent

agent_config = DQNConfig()
agent_config

Training command used for experiments:

```powershell
cd D:\AirSim\rl_drone_navigation
conda activate airsim-rl
python src\train_dqn.py --scenario blocks --episodes 200 --target-x 20 --target-y 0 --target-z -3
python src\train_ppo.py --scenario blocks --episodes 200 --target-x 20 --target-y 0 --target-z -3
```

Evaluation command:

```powershell
python src\evaluate.py --algorithm dqn --scenario blocks --episodes 20
python src\evaluate.py --algorithm ppo --scenario blocks --episodes 20
```

## 5. Results

This section loads results produced by `train_dqn.py`, `train_ppo.py`, and `evaluate.py`. Change `SCENARIO` and `ALGORITHM` to inspect a different experiment folder.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

SCENARIO = "blocks"
ALGORITHM = "dqn"  # change to "ppo" for PPO results
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / SCENARIO / ALGORITHM
training_log = EXPERIMENT_DIR / "results" / "training_log.csv"

def load_csv(path):
    if not path.exists():
        return []
    with path.open("r", newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

train_rows = load_csv(training_log)
len(train_rows), training_log

In [ ]:
def column(rows, key):
    values = []
    for row in rows:
        try:
            values.append(float(row[key]))
        except (KeyError, ValueError):
            values.append(np.nan)
    return np.array(values, dtype=float)

def moving_average(values, window=10):
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode="valid")

if train_rows:
    episodes = column(train_rows, "episode")
    reward = column(train_rows, "reward")
    success = column(train_rows, "success")
    collision = column(train_rows, "collision")
    distance = column(train_rows, "final_distance")

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.ravel()
    axes[0].plot(episodes, reward, alpha=0.4)
    ma = moving_average(reward)
    axes[0].plot(episodes[-len(ma):], ma)
    axes[0].set_title("Episode Reward")

    ma = moving_average(success)
    axes[1].plot(episodes[-len(ma):], ma)
    axes[1].set_title("Success Rate")
    axes[1].set_ylim(-0.05, 1.05)

    ma = moving_average(collision)
    axes[2].plot(episodes[-len(ma):], ma)
    axes[2].set_title("Collision Rate")
    axes[2].set_ylim(-0.05, 1.05)

    axes[3].plot(episodes, distance)
    axes[3].set_title("Final Distance to Target")
    for ax in axes:
        ax.set_xlabel("Episode")
    fig.tight_layout()
else:
    print("No training log found yet. Run training first, then rerun this cell.")

In [ ]:
eval_log = EXPERIMENT_DIR / "results" / "evaluation_log.csv"
eval_rows = load_csv(eval_log)

if eval_rows:
    rewards = column(eval_rows, "reward")
    success = column(eval_rows, "success")
    collision = column(eval_rows, "collision")
    steps = column(eval_rows, "steps")
    final_distance = column(eval_rows, "final_distance")

    evaluation_summary = {
        "episodes": len(eval_rows),
        "success_rate": float(np.nanmean(success)),
        "collision_rate": float(np.nanmean(collision)),
        "average_reward": float(np.nanmean(rewards)),
        "average_steps": float(np.nanmean(steps)),
        "average_final_distance": float(np.nanmean(final_distance)),
    }
    evaluation_summary
else:
    print("No evaluation log found yet. Run evaluation first, then rerun this cell.")

### Result Discussion Template

After running experiments, replace this paragraph with the final observed values. Discuss whether reward increased during training, whether success rate improved, whether collisions decreased, and whether the final policy reached the target more efficiently than early exploratory episodes.

## 6. Discussion

The DQN and PPO baselines are useful complementary approaches because both map visual observations and motion state to discrete flight decisions, but they optimise different objectives. A successful run should show increasing episode rewards, decreasing final distance to the target, and a lower collision rate over time.

Potential strengths:

- Simple and explainable discrete action policy.
- Uses visual input rather than only ground-truth coordinates.
- Produces clear quantitative metrics for evaluation.

Limitations:

- DQN can be sample inefficient in visual environments, while PPO may require careful rollout and learning-rate tuning.
- The action space is simplified compared with real continuous drone control.
- A policy trained in Blocks may overfit to one environment layout.
- Sim-to-real transfer is not addressed in this baseline.

Future work could compare SAC, use domain randomization, train across multiple obstacle layouts, add curriculum learning, and evaluate generalization to unseen target positions.

## 7. References

1. Shah, S., Dey, D., Lovett, C., and Kapoor, A. AirSim: High-Fidelity Visual and Physical Simulation for Autonomous Vehicles. Field and Service Robotics, 2018.
2. Mnih, V. et al. Human-level control through deep reinforcement learning. Nature, 2015.
3. Schulman, J. et al. Proximal Policy Optimization Algorithms. arXiv:1707.06347, 2017.